# GTEx whole-slide images, opened cold

This notebook opens the store using nothing but the store itself: no ingest code, no DICOM reader, and no copy of the slides. It needs `zarr`, `xarray` and `icechunk`, plus `arraylake` to open the published repo. Every pixel it shows is read from IDC's public bucket, `s3://idc-open-data`, through the store's references.

The store holds every GTEx slide that NCI Imaging Data Commons (IDC) publishes: H&E sections of normal tissue from GTEx donors, as OME-Zarr 0.6 images. `STORE` is the published repo, or a local build's path.

In [ ]:
import json
import os
from collections import Counter

import icechunk
import xarray as xr
import zarr

STORE = os.environ.get("IDC_GTEX_STORE", "arraylake://earthmover-biology/idc-gtex")

if STORE.startswith("arraylake://"):
    # The published repo lives in Arraylake, whose client applies its virtual chunk containers.
    from arraylake import Client

    repo = Client().get_repo(STORE.removeprefix("arraylake://"))
else:
    # The slides' bytes stay in IDC's bucket, which anyone may read without credentials.
    access = {"s3://idc-open-data/": icechunk.Credentials.S3(icechunk.s3_anonymous_credentials())}
    repo = icechunk.Repository.open(icechunk.local_filesystem_storage(STORE), authorize_virtual_chunk_access=access)
session = repo.readonly_session("main")
root = zarr.open_group(session.store, mode="r")

print(root.attrs["title"])
print("IDC release:", root.attrs["idc_version"])
print("codecs a reader needs:", root.attrs["codecs"])
print("slides left out:", len(root.attrs["left_out"]))

## Donors and their slides

The hierarchy is `gtex/<donor>/<slide>`, named as IDC names them: a donor is a GTEx subject ID, a slide is the barcode on its glass. Each slide group says what tissue it shows.

In [ ]:
gtex = root["gtex"]
print(gtex.attrs["license"], "·", gtex.attrs["source_DOI"])

slides = {path: group for path, group in gtex.members(max_depth=1) if path.count("/") == 1}
donors = {path.split("/")[0] for path in slides}
print(f"{len(donors)} donors, {len(slides)} slides")

tissues = Counter(group.attrs["specimen"]["tissue"] for group in slides.values())
for tissue, n in tissues.most_common(8):
    print(f"{n:6d}  {tissue}")

## One slide

A slide's `image` is the scanner's pyramid: level 0 at full resolution, each further level a coarser copy. The `scale` says how big a pixel is, in micrometers. Each chunk is one 240 x 240 JPEG tile of the slide's DICOM file.

zarr refuses to open an array whose codec it cannot load, so this cell reads each level's `zarr.json` document directly: its shape, chunks and codecs are plain JSON.

In [ ]:
from zarr.core.buffer import default_buffer_prototype


async def array_metadata(key):
    return json.loads((await session.store.get(f"{key}/zarr.json", prototype=default_buffer_prototype())).to_bytes())


path, slide = sorted(slides.items())[0]
print(path, slide.attrs["specimen"])

image = slide["image"]
multiscale = image.attrs["ome"]["multiscales"][0]
for dataset in multiscale["datasets"]:
    level = await array_metadata(f"gtex/{path}/image/{dataset['path']}")
    scale = dataset["coordinateTransformations"][0]["transformations"][0]["scale"]
    chunks = level["chunk_grid"]["configuration"]["chunk_shape"]
    codecs = [c["name"] for c in level["codecs"]]
    print(f"level {dataset['path']}: {level['shape']}, {scale[1]:.3f} µm pixels, chunks {chunks}, {codecs}")

The thumbnail is one uncompressed frame, so it decodes with `zarr` alone. With its scale, `xarray` labels both axes in micrometers.

In [ ]:
thumbnail = slide["thumbnail"]
# One level and no translation: the dataset's transformation is the scale itself.
scale = thumbnail.attrs["ome"]["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]["scale"]
pixels = thumbnail["0"][:]
rgb = xr.DataArray(
    pixels,
    dims=("c", "y", "x"),
    coords={"y": [i * scale[1] for i in range(pixels.shape[1])], "x": [i * scale[2] for i in range(pixels.shape[2])]},
    attrs={"units": "micrometer"},
)
print(rgb.sizes, f"spanning {float(rgb.y[-1]) / 1000:.1f} mm by {float(rgb.x[-1]) / 1000:.1f} mm")
rgb.mean(dim=("y", "x")).values

## One full-resolution tile

Level 0's chunks are JPEG tiles. Their stored bytes come back from the store as they are in the DICOM file, and a browser shows them as they are. Decoding them into arrays needs the `imagecodecs_jpeg` codec, which the `imagecodecs-zarr` package registers; with it installed, `image["0"][:, y0:y1, x0:x1]` returns pixels.

In [ ]:
from IPython.display import Image

level = await array_metadata(f"gtex/{path}/image/0")
(_, height, width), (_, ty, tx) = level["shape"], level["chunk_grid"]["configuration"]["chunk_shape"]
rows, cols = -(-height // ty), -(-width // tx)
tile = await session.store.get(f"gtex/{path}/image/0/c/0/{rows // 2}/{cols // 2}", prototype=default_buffer_prototype())
print(f"tile ({rows // 2}, {cols // 2}) of {rows} x {cols}: {len(tile.to_bytes()):,} bytes of JPEG")
Image(data=tile.to_bytes(), format="jpeg")

## What the DICOM said

Every instance's DICOM header travels with the slide, in `dicom`: one JSON document keyed by the store node that holds the instance's pixels.

In [ ]:
headers = json.loads(bytes(slide["dicom"][:]))
level0 = headers["image/0"]
for tag, name in [("00080070", "Manufacturer"), ("00081090", "ManufacturerModelName"),
                  ("00480006", "TotalPixelMatrixColumns"), ("00480007", "TotalPixelMatrixRows"),
                  ("00280008", "NumberOfFrames"), ("00480001", "ImagedVolumeWidth")]:
    print(f"{name:24s} {level0.get(tag, {}).get('Value')}")
print("instances:", sorted(headers))